<a href="https://colab.research.google.com/github/swxjj/swxjj.github.io/blob/main/AppPA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import pandas as pd
import requests
import streamlit as st

# 1. Hacemos el pedido a la API
api = "https://apis.datos.gob.ar/series/api/series/?ids=150.1_CSTA_BATAL_0_D_20&format=json&limit=5000&start_date=2023-11-01"
respuesta = requests.get(api)
datos = respuesta.json()

# 2. EL TRUCO MAGISTRAL:
# Le decimos a Pandas que solo mire adentro de 'data' y de paso ¡bautizamos las columnas!
df_canasta = pd.DataFrame(datos['data'], columns=['fecha', 'Costo_Canasta'])

# 3. Acomodamos las fechas para que los gráficos salgan bien (Regla de oro)
df_canasta['fecha'] = pd.to_datetime(df_canasta['fecha'])
df_canasta.set_index('fecha', inplace=True)

print(df_canasta.head())

            Costo_Canasta
fecha                    
2023-11-01      126361.27
2023-12-01      160452.53
2024-01-01      193146.66
2024-02-01      223592.74
2024-03-01      250286.44


In [81]:
%%writefile app_inflacion.py
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf
import streamlit as st

@st.cache_data
def load_data():
  api=f"https://apis.datos.gob.ar/series/api/series/?ids=150.1_CSTA_BATAL_0_D_20&format=json&limit=5000&start_date=2023-11-01"
  respuesta=requests.get(api)
  datos=respuesta.json()
  df_canasta = pd.DataFrame(datos['data'], columns=['fecha', 'Costo_Canasta'])
  df_canasta['fecha'] = pd.to_datetime(df_canasta['fecha'])
  df_canasta.set_index('fecha', inplace=True)
  return df_canasta

with st.spinner("Descargando datos oficiales del INDEC..."):
    df = load_data()
st.title("Análisis del Poder Adquisitivo según Clase Social")
# input "asigned" social class
clase_s = st.selectbox("¿Con qué clase social se identifica?", ['Alta', 'Media', 'Baja', 'No sé'], index=None, placeholder="Elija una opción")
# creo variable vacía que guarde la elección
clase_final = ""
# display the selected class
if clase_s is None:
    st.write("Debe elegir una clase social")
elif clase_s != 'No sé':
    clase_final = clase_s
else:
    salario = st.number_input("Ingrese su salario familiar estimado", min_value=100000, value=500000, step=100000)
    personas = st.number_input("Ingrese la cantidad de personas en su hogar", min_value=1, value=1, step=1)
    if salario/personas > df['Costo_Canasta'].max()*1.5:
        st.write(f"Su clase es Alta")
        clase_final = "Alta"
    elif salario/personas < df['Costo_Canasta'].max():
        st.write(f"Su clase es Baja")
        clase_final = "Baja"
    else:
        st.write(f"Su clase es Media")
        clase_final = "Media"
# display the selected class

st.button("Generar gráfico")






Overwriting app_inflacion.py


In [ ]:
from pyngrok import ngrok
import subprocess

# Reemplaza con tu token real
ngrok.set_auth_token("3AjcUqpLRgGmCIE8skqFj7XsKKx_52UHBLKJ3ggCLiuDJd28V")
ngrok.kill()

# Corremos el nuevo archivo app_inflacion.py
proceso = subprocess.Popen(["streamlit", "run", "app_inflacion.py"])
public_url = ngrok.connect(8501)
print(f"👉 HAZ CLIC AQUÍ PARA VER TU NUEVA APP: {public_url}")

👉 HAZ CLIC AQUÍ PARA VER TU NUEVA APP: NgrokTunnel: "https://ila-profiction-practisedly.ngrok-free.dev" -> "http://localhost:8501"
